# Laguna XS.2 — Clean End-to-End Causal Expert Surgery
## AWS g7e.2xlarge · RTX PRO 6000 Blackwell 96GB · 8 vCPU · 64 GiB RAM

This notebook is rebuilt **from scratch**.

It uses one external CSV consistently across all phases:

```text
selection target/control
        ↓
causal layer localization
        ↓
hierarchical expert search
        ↓
individual validation
        ↓
bootstrap + renormalization
        ↓
causal expert
        ↓
global routing measurement on selection targets
        ↓
most-routed expert
        ↓
matched one-expert training
        ↓
heldout target/control evaluation
```

Matched training arms:

1. strongest causal expert;
2. globally most-routed expert;
3. random global expert;
4. random expert from the causal expert's layer.

All arms train **exactly one Laguna routed expert block**:

\[
3,145,728 \text{ trainable parameters}
\]

with the same training examples, optimizer, learning rate, gradient
accumulation and update budget.

The purpose is to test:

\[
\boxed{
\text{Does causal expert selection outperform routing-frequency selection
at a matched parameter and data budget?}
}
\]

## 1 — Install dependencies

In [2]:
# Keep the CUDA-enabled PyTorch build supplied by the instance.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2 — Runtime tuning for g7e.2xlarge

In [3]:
import os

# 8 vCPU host: leave headroom for Python / I/O.
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "4"

# 64 GiB RAM: conservative checkpoint-loading parallelism.
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

# CUDA.
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for AWS g7e.2xlarge.")

Runtime configured for AWS g7e.2xlarge.


## 3 — Hardware and storage preflight

In [4]:
import os
import shutil
import platform
from pathlib import Path

import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary or larger).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than a 64-GiB-class host detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen_devices = set()

for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

=== Host ===
Python: 3.10.12
Logical CPUs: 8
RAM total:     62.27 GiB
RAM available: 60.86 GiB

=== CUDA ===
Torch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
Compute capability: (12, 0)

=== Storage ===
Work root: /home/ec2-user/workspace
Disk free: 918.33 GiB
Hardware preflight: PASS


## 4 — Load the experiment CSV

Expected schema:

```text
split,kind,prompt,reference
```

Required minimum counts:

```text
selection / target   50
selection / control  50
train     / target   50
heldout   / target   50
heldout   / control  50
```

Lookup order:

1. `LAGUNA_EXPERIMENT_CSV`
2. `/home/ec2-user/workspace/laguna_frontend_experiment.csv`
3. `/workspace/laguna_frontend_experiment.csv`
4. `/mnt/data/laguna_frontend_experiment.csv`

In [5]:
import pandas as pd
import numpy as np

candidates = []

if os.environ.get("LAGUNA_EXPERIMENT_CSV"):
    candidates.append(
        Path(os.environ["LAGUNA_EXPERIMENT_CSV"]).expanduser()
    )

candidates.extend([
    Path("/home/ec2-user/workspace/laguna_frontend_experiment.csv"),
    Path("/workspace/laguna_frontend_experiment.csv"),
    Path("/mnt/data/laguna_frontend_experiment.csv"),
])

EXPERIMENT_CSV = next(
    (p.resolve() for p in candidates if p.exists()),
    None,
)

if EXPERIMENT_CSV is None:
    raise FileNotFoundError(
        "laguna_frontend_experiment.csv not found. "
        "Set LAGUNA_EXPERIMENT_CSV to the full file path."
    )

experiment_df = pd.read_csv(EXPERIMENT_CSV)

required_cols = {"split", "kind", "prompt", "reference"}
missing_cols = required_cols - set(experiment_df.columns)

if missing_cols:
    raise ValueError(
        f"Experiment CSV missing columns: {sorted(missing_cols)}"
    )

experiment_df["split"] = (
    experiment_df["split"].astype(str).str.lower().str.strip()
)
experiment_df["kind"] = (
    experiment_df["kind"].astype(str).str.lower().str.strip()
)

allowed_splits = {"selection", "train", "heldout"}
allowed_kinds = {"target", "control"}

bad_splits = sorted(set(experiment_df["split"]) - allowed_splits)
bad_kinds = sorted(set(experiment_df["kind"]) - allowed_kinds)

if bad_splits:
    raise ValueError(f"Unsupported split labels: {bad_splits}")

if bad_kinds:
    raise ValueError(f"Unsupported kind labels: {bad_kinds}")

minimums = {
    ("selection", "target"): 50,
    ("selection", "control"): 50,
    ("train", "target"): 50,
    ("heldout", "target"): 50,
    ("heldout", "control"): 50,
}

too_small = []

for (split, kind), minimum in minimums.items():
    actual = len(
        experiment_df[
            (experiment_df["split"] == split)
            & (experiment_df["kind"] == kind)
        ]
    )

    if actual < minimum:
        too_small.append((split, kind, actual, minimum))

if too_small:
    raise RuntimeError(
        "Dataset is below required minimums:\n"
        + "\n".join(
            f"{s}/{k}: {a} < {m}"
            for s, k, a, m in too_small
        )
    )

selection_df = experiment_df[
    experiment_df["split"] == "selection"
].reset_index(drop=True)

train_target_df = experiment_df[
    (experiment_df["split"] == "train")
    & (experiment_df["kind"] == "target")
].reset_index(drop=True)

heldout_df = experiment_df[
    experiment_df["split"] == "heldout"
].reset_index(drop=True)

print("Experiment CSV:", EXPERIMENT_CSV)

display(
    experiment_df.groupby(["split", "kind"])
    .size()
    .rename("count")
    .reset_index()
)

Experiment CSV: /home/ec2-user/workspace/laguna_frontend_experiment.csv


,split,kind,count
0,heldout,control,50
1,heldout,target,50
2,selection,control,50
3,selection,target,50
4,train,target,50


### Split leakage check

In [6]:
selection_prompts = set(selection_df["prompt"].astype(str))
train_prompts = set(train_target_df["prompt"].astype(str))
heldout_prompts = set(heldout_df["prompt"].astype(str))

leaks = {
    "selection_vs_train": selection_prompts & train_prompts,
    "selection_vs_heldout": selection_prompts & heldout_prompts,
    "train_vs_heldout": train_prompts & heldout_prompts,
}

for name, overlap in leaks.items():
    print(name, "overlap:", len(overlap))

if any(leaks.values()):
    raise RuntimeError("Prompt leakage detected across splits.")

print("Split leakage check: PASS")

selection_vs_train overlap: 0
selection_vs_heldout overlap: 0
train_vs_heldout overlap: 0
Split leakage check: PASS


## 5 — Resolve the official BF16 Laguna XS.2 checkpoint

In [7]:
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)

    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB."
        )

    print("Downloading Laguna XS.2 BF16...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.py",
            "*.jinja",
            "LICENSE*",
            "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]

if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print("Checkpoint verification: PASS")

MODEL_PATH: /home/ec2-user/workspace/models/Laguna-XS.2
14-shard payload: 66.889 GB
Checkpoint verification: PASS


/home/ec2-user/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6 — Register Laguna checkpoint conversion mapping

In [8]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose the native Laguna checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

Laguna checkpoint conversion mapping: REGISTERED
Conversion operations: 4


## 7 — Load BF16 model directly onto the RTX PRO 6000

In [9]:
import gc
import time
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if (
        ".mlp.experts" in k
        or ".mlp.gate" in k
        or "e_score_correction_bias" in k
    )
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")
print("BF16 load: PASS")

Transformers: 5.14.1


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 639/639 [08:26<00:00,  1.26it/s]


Loaded in 8.51 min
GPU allocated: 62.29 GiB
GPU reserved:  89.38 GiB
GPU peak:      63.29 GiB
Driver free:   5.04 GiB
Host RAM available: 58.79 GiB
BF16 load: PASS


## 8 — Validate Laguna MoE architecture

In [10]:
cfg = model.config

SPARSE_LAYERS = []

for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)

    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample_mlp = model.model.layers[SPARSE_LAYERS[0]].mlp

assert tuple(sample_mlp.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample_mlp.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample_mlp.experts.gate_up_proj[0].numel()
    + sample_mlp.experts.down_proj[0].numel()
)

print("Sparse layers:", SPARSE_LAYERS)
print(f"Params / expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("Architecture validation: PASS")

Sparse layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
Params / expert: 3,145,728 (3.146M)
Architecture validation: PASS


## 9 — Correct Laguna teacher-forcing format

The no-thinking assistant prefix is followed by a **newline** before the
reference answer:

```text
<assistant>
</think>
REFERENCE
```

In [11]:
def chat_prefix_text(prompt):
    messages = [{"role": "user", "content": prompt}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0

    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(
            f"Could not identify answer boundary for: {prompt!r}"
        )

    return full_ids, start

## 10 — Generic aligned scoring batch

In [12]:
def build_scoring_batch(df):
    df = df.reset_index(drop=True).copy()

    parsed = [
        parse_case(r.prompt, r.reference)
        for r in df.itertuples(index=False)
    ]

    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    targets = torch.full(
        (batch_size, max_ref),
        -100,
        dtype=torch.long,
    )

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1

        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

@torch.inference_mode()
def score_batch(batch):
    import torch.nn.functional as F

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    per_example = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example

    return result

## 11 — Build selection and held-out batches

In [13]:
SELECTION_BATCH = build_scoring_batch(selection_df)
HELDOUT_BATCH = build_scoring_batch(heldout_df)

SELECTION_BASE_NLL = score_batch(SELECTION_BATCH)
HELDOUT_BASE_NLL = score_batch(HELDOUT_BATCH)

selection_target_mask = selection_df["kind"].values == "target"
selection_control_mask = selection_df["kind"].values == "control"

heldout_target_mask = heldout_df["kind"].values == "target"
heldout_control_mask = heldout_df["kind"].values == "control"

print("Selection baseline target NLL :", float(SELECTION_BASE_NLL[selection_target_mask].mean()))
print("Selection baseline control NLL:", float(SELECTION_BASE_NLL[selection_control_mask].mean()))

print("Heldout baseline target NLL :", float(HELDOUT_BASE_NLL[heldout_target_mask].mean()))
print("Heldout baseline control NLL:", float(HELDOUT_BASE_NLL[heldout_control_mask].mean()))

Selection baseline target NLL : 5.2584381103515625
Selection baseline control NLL: 4.7993879318237305
Heldout baseline target NLL : 4.762903213500977
Heldout baseline control NLL: 4.120146751403809


## 12 — Scoring sanity check

In [14]:
demo_prefix = chat_prefix_text(selection_df.iloc[0]["prompt"])

print("Prefix tail:", repr(demo_prefix[-60:]))
print(
    "Teacher-forced boundary:",
    repr(
        (
            demo_prefix
            + "\n"
            + selection_df.iloc[0]["reference"]
        )[-90:]
    )
)

b = 0

ids = SELECTION_BATCH["input_ids"][b:b+1]
mask = SELECTION_BATCH["attention_mask"][b:b+1]
pos = SELECTION_BATCH["position_ids"][b:b+1]
keep = SELECTION_BATCH["pred_positions"]

with torch.inference_mode():
    selected_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=keep,
        return_dict=True,
    ).logits.float()

    full_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=0,
        return_dict=True,
    ).logits[:, keep, :].float()

max_diff = float(
    (selected_logits - full_logits).abs().max().item()
)

print("max |selective - full sliced logits|:", max_diff)

if max_diff > 1e-4:
    raise RuntimeError(
        "Selective-logit scorer does not match full-logit scoring."
    )

del selected_logits, full_logits
torch.cuda.empty_cache()

print("Scoring sanity: PASS")

Prefix tail: 'nk below its min-content width?\n</user>\n<assistant>\n</think>'
Teacher-forced boundary: 'ets a child shrink below its min-content width?\n</user>\n<assistant>\n</think>\nmin-width: 0;'
max |selective - full sliced logits|: 0.0
Scoring sanity: PASS


## 13 — Fixed-routing causal intervention

In [15]:
from contextlib import contextmanager
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)

    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")

    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(
    layer_idx,
    expert_ids=None,
    zero_all_routed=False,
    renormalize=False,
):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward

    expert_ids = (
        []
        if expert_ids is None
        else [int(x) for x in expert_ids]
    )

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = (
            original_forward(hidden_states)
        )

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )

            keep = ~torch.isin(selected_experts, ids)

            routing_weights = (
                routing_weights
                * keep.to(routing_weights.dtype)
            )

            if renormalize:
                denom = routing_weights.sum(
                    dim=-1,
                    keepdim=True,
                )

                routing_weights = torch.where(
                    denom > 0,
                    routing_weights
                    / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return (
            router_logits,
            routing_weights,
            selected_experts,
        )

    gate.forward = types.MethodType(
        patched_forward,
        gate,
    )

    try:
        yield
    finally:
        gate.forward = original_forward

## 14 — Causal score definition

In [16]:
CONTROL_PENALTY = 0.75

def summarize_selection_delta(ablated_nll):
    delta = np.asarray(ablated_nll) - SELECTION_BASE_NLL

    target_delta = float(
        delta[selection_target_mask].mean()
    )

    control_delta = float(
        delta[selection_control_mask].mean()
    )

    causal_specificity = (
        target_delta
        - CONTROL_PENALTY
        * max(control_delta, 0.0)
    )

    return {
        "target_delta_nll": target_delta,
        "control_delta_nll": control_delta,
        "causal_specificity": causal_specificity,
        "per_example_delta": delta,
    }

## 15 — Causal layer sweep

In [17]:
from tqdm.auto import tqdm
import time

RESULTS = WORK_ROOT / "laguna_xs2_v5_results"
RESULTS.mkdir(parents=True, exist_ok=True)

layer_rows = []

t0 = time.time()

for layer_idx in tqdm(
    SPARSE_LAYERS,
    desc="39-layer causal sweep",
):
    with gate_intervention(
        layer_idx,
        zero_all_routed=True,
    ):
        nll = score_batch(SELECTION_BATCH)

    m = summarize_selection_delta(nll)

    layer_rows.append({
        "layer": int(layer_idx),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

layer_df = pd.DataFrame(layer_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

layer_df.to_csv(
    RESULTS / "layer_causal_scores.csv",
    index=False,
)

print(
    f"Layer sweep time: {(time.time()-t0)/60:.2f} min"
)

display(layer_df.head(15))

39-layer causal sweep: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:18<00:00,  2.09it/s]

Layer sweep time: 0.31 min


,layer,target_delta_nll,control_delta_nll,causal_specificity
0,36,1.238812,0.218739,1.074758
1,38,0.351376,0.067112,0.301041
2,33,0.255101,0.082050,0.193563
3,27,0.240993,0.108592,0.159549
4,25,0.561571,0.564287,0.138355
5,24,0.061678,-0.067504,0.061678
6,16,0.189044,0.193727,0.043749
7,8,0.032251,0.011593,0.023556
8,18,0.056368,0.059783,0.011531
9,3,-0.029724,-0.082749,-0.029724


## 16 — Hierarchical expert search

In [18]:
def intervention_score(
    layer_idx,
    expert_ids,
    renormalize=False,
):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_batch(SELECTION_BATCH)

    return summarize_selection_delta(nll)

def hierarchical_expert_search(
    layer_idx,
    seed=17,
    initial_group_size=32,
    beam_width=3,
):
    rng = np.random.default_rng(seed)

    order = rng.permutation(
        cfg.num_experts
    ).tolist()

    frontier = [
        order[i:i+initial_group_size]
        for i in range(
            0,
            len(order),
            initial_group_size,
        )
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in tqdm(
            frontier,
            desc=f"L{layer_idx} level {level}",
            leave=False,
        ):
            m = intervention_score(
                layer_idx,
                group,
            )

            rec = {
                "layer": int(layer_idx),
                "seed": int(seed),
                "level": int(level),
                "group_size": len(group),
                "experts": list(map(int, group)),
                "target_delta_nll": m["target_delta_nll"],
                "control_delta_nll": m["control_delta_nll"],
                "causal_specificity": m["causal_specificity"],
            }

            history.append(rec)
            current.append(rec)

        current.sort(
            key=lambda x: x["causal_specificity"],
            reverse=True,
        )

        keep = current[:beam_width]

        if all(
            x["group_size"] == 1
            for x in keep
        ):
            break

        nxt = []

        for rec in keep:
            g = rec["experts"]

            if len(g) == 1:
                nxt.append(g)
            else:
                mid = len(g) // 2
                nxt.extend([
                    g[:mid],
                    g[mid:],
                ])

        frontier = [
            x for x in nxt if x
        ]

        level += 1

    hist = pd.DataFrame(history)

    leaf_size = hist["group_size"].min()

    leaves = hist[
        hist["group_size"] == leaf_size
    ].sort_values(
        "causal_specificity",
        ascending=False,
    )

    return hist, leaves

## 17 — Search top four causal layers with two randomized partitions

In [19]:
TOP_LAYERS = 4
SEARCH_SEEDS = [17, 53]
INITIAL_GROUP_SIZE = 32
BEAM_WIDTH = 3

candidate_layers = (
    layer_df.head(TOP_LAYERS)["layer"]
    .astype(int)
    .tolist()
)

print("Candidate layers:", candidate_layers)

histories = []
leaf_frames = []

for layer_idx in candidate_layers:
    for seed in SEARCH_SEEDS:
        hist, leaves = hierarchical_expert_search(
            layer_idx,
            seed=seed,
            initial_group_size=INITIAL_GROUP_SIZE,
            beam_width=BEAM_WIDTH,
        )

        histories.append(hist)
        leaf_frames.append(leaves)

group_history = pd.concat(
    histories,
    ignore_index=True,
)

leaf_df = pd.concat(
    leaf_frames,
    ignore_index=True,
)

group_history.to_json(
    RESULTS / "hierarchical_group_history.json",
    orient="records",
    indent=2,
)

display(leaf_df.head(30))

Candidate layers: [36, 38, 33, 27]


,layer,seed,level,group_size,experts,target_delta_nll,control_delta_nll,causal_specificity
0,36,17,5,1,[229],1.280375e+00,4.886748e-01,0.913869
1,36,17,5,1,[95],1.858674e-02,-4.808240e-03,0.018587
2,36,17,5,1,[13],1.006799e-02,-3.219644e-03,0.010068
3,36,17,5,1,[204],-9.918213e-07,7.629394e-08,-0.000001
4,36,17,5,1,[189],-3.347473e-03,-1.624335e-03,-0.003347
5,36,17,5,1,[254],-8.092509e-03,6.375506e-05,-0.008140
6,36,53,5,1,[229],1.280375e+00,4.886748e-01,0.913869
7,36,53,5,1,[95],1.858674e-02,-4.808240e-03,0.018587
8,36,53,5,1,[122],-6.733952e-03,-4.286220e-03,-0.006734
9,36,53,5,1,[102],-5.720141e-03,2.170917e-03,-0.007348


## 18 — Exact individual expert validation

In [20]:
leaf_pairs = sorted({
    (int(r.layer), int(e))
    for r in leaf_df.itertuples(index=False)
    for e in r.experts
})

print("Unique leaf candidates:", len(leaf_pairs))

individual_rows = []

for layer_idx, expert_id in tqdm(
    leaf_pairs,
    desc="Individual validation",
):
    m = intervention_score(
        layer_idx,
        [expert_id],
    )

    individual_rows.append({
        "layer": int(layer_idx),
        "expert": int(expert_id),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"].tolist(),
    })

individual_df = pd.DataFrame(
    individual_rows
).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

display(individual_df.head(25))

Unique leaf candidates: 40


Individual validation: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:19<00:00,  2.09it/s]


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,per_example_delta
0,36,229,1.280375e+00,4.886748e-01,0.913869,"[-0.016780614852905273, 1.6728472709655762, 2...."
1,38,60,2.445150e-01,4.952861e-02,0.207369,"[0.0121612548828125, 0.32450056076049805, 1.86..."
2,27,146,2.360537e-01,1.804479e-01,0.100718,"[0.03569364547729492, -0.19040679931640625, 0...."
3,33,166,7.009730e-02,-1.154884e-02,0.070097,"[0.011115074157714844, 0.03645682334899902, -0..."
4,33,206,5.898509e-02,1.513472e-03,0.057850,"[0.08012127876281738, -0.07839012145996094, -0..."
5,27,192,4.630700e-02,-8.726914e-03,0.046307,"[-0.048911333084106445, 0.05187511444091797, 0..."
6,33,234,4.472505e-02,-1.131347e-02,0.044725,"[0.06992411613464355, 0.16161847114562988, -0...."
7,27,40,6.347551e-02,4.123253e-02,0.032551,"[0.06931805610656738, 0.03368949890136719, 0.0..."
8,27,118,2.451584e-02,-3.806626e-02,0.024516,"[-0.008538246154785156, -0.05396604537963867, ..."
9,38,168,2.050057e-02,-2.181269e-03,0.020501,"[0.030826807022094727, -0.09655070304870605, 0..."


## 19 — Bootstrap confidence intervals

In [21]:
def bootstrap_specificity(
    per_example_delta,
    n_boot=5000,
    seed=123,
):
    rng = np.random.default_rng(seed)

    d = np.asarray(
        per_example_delta,
        dtype=np.float64,
    )

    t = d[selection_target_mask]
    c = d[selection_control_mask]

    vals = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for i in range(n_boot):
        tb = rng.choice(
            t,
            size=len(t),
            replace=True,
        ).mean()

        cb = rng.choice(
            c,
            size=len(c),
            replace=True,
        ).mean()

        vals[i] = (
            tb
            - CONTROL_PENALTY
            * max(cb, 0.0)
        )

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

boot_rows = []

for r in individual_df.itertuples(index=False):
    boot_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        **bootstrap_specificity(
            r.per_example_delta
        ),
    })

boot_df = pd.DataFrame(boot_rows)

final_df = individual_df.merge(
    boot_df,
    on=["layer", "expert"],
    how="left",
).sort_values(
    ["p_positive", "causal_specificity"],
    ascending=False,
).reset_index(drop=True)

display(
    final_df[
        [
            "layer",
            "expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "p_positive",
        ]
    ].head(25)
)

,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,ci_2.5,ci_97.5,p_positive
0,36,229,1.280375,0.488675,0.913869,0.411105,1.435714,1.0000
1,38,60,0.244515,0.049529,0.207369,0.062659,0.400115,0.9994
2,33,166,0.070097,-0.011549,0.070097,0.026357,0.107441,0.9990
3,27,146,0.236054,0.180448,0.100718,0.031798,0.171312,0.9986
4,33,206,0.058985,0.001513,0.057850,0.013595,0.089700,0.9954
5,33,234,0.044725,-0.011313,0.044725,-0.007953,0.094255,0.9450
6,27,192,0.046307,-0.008727,0.046307,-0.009265,0.091065,0.9442
7,27,118,0.024516,-0.038066,0.024516,-0.015447,0.065351,0.8794
8,27,40,0.063476,0.041233,0.032551,-0.023400,0.087272,0.8758
9,38,210,0.016306,-0.000034,0.016306,-0.007969,0.049289,0.8700


## 20 — Renormalization robustness

In [22]:
ROBUST_TOP_N = min(
    16,
    len(final_df),
)

robust_rows = []

for r in tqdm(
    list(
        final_df.head(
            ROBUST_TOP_N
        ).itertuples(index=False)
    ),
    desc="Renormalized validation",
):
    m = intervention_score(
        int(r.layer),
        [int(r.expert)],
        renormalize=True,
    )

    robust_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        "renorm_target_delta_nll": m["target_delta_nll"],
        "renorm_control_delta_nll": m["control_delta_nll"],
        "renorm_causal_specificity": m["causal_specificity"],
    })

robust_df = pd.DataFrame(robust_rows)

final_robust = final_df.merge(
    robust_df,
    on=["layer", "expert"],
    how="left",
)

final_robust.drop(
    columns=["per_example_delta"],
).to_csv(
    RESULTS / "causal_candidates.csv",
    index=False,
)

display(
    final_robust[
        [
            "layer",
            "expert",
            "causal_specificity",
            "renorm_causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "p_positive",
        ]
    ].head(20)
)

Renormalized validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:07<00:00,  2.09it/s]


,layer,expert,causal_specificity,renorm_causal_specificity,ci_2.5,ci_97.5,p_positive
0,36,229,0.913869,0.904908,0.411105,1.435714,1.0000
1,38,60,0.207369,0.211055,0.062659,0.400115,0.9994
2,33,166,0.070097,0.080556,0.026357,0.107441,0.9990
3,27,146,0.100718,0.088627,0.031798,0.171312,0.9986
4,33,206,0.057850,0.032775,0.013595,0.089700,0.9954
5,33,234,0.044725,0.020106,-0.007953,0.094255,0.9450
6,27,192,0.046307,0.042701,-0.009265,0.091065,0.9442
7,27,118,0.024516,0.047576,-0.015447,0.065351,0.8794
8,27,40,0.032551,0.035497,-0.023400,0.087272,0.8758
9,38,210,0.016306,0.021737,-0.007969,0.049289,0.8700


## 21 — Strict causal expert selector

In [23]:
strict_candidates = final_robust[
    (final_robust["ci_2.5"] > 0)
    & (final_robust["causal_specificity"] > 0)
    & (final_robust["renorm_causal_specificity"] > 0)
].copy()

if strict_candidates.empty:
    raise RuntimeError(
        "No expert passed strict causal selection."
    )

strict_candidates = strict_candidates.sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

CAUSAL_PAIR = (
    int(strict_candidates.iloc[0]["layer"]),
    int(strict_candidates.iloc[0]["expert"]),
)

print("CAUSAL_PAIR =", CAUSAL_PAIR)

display(
    strict_candidates[
        [
            "layer",
            "expert",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "renorm_causal_specificity",
        ]
    ].head(10)
)

CAUSAL_PAIR = (36, 229)


,layer,expert,causal_specificity,ci_2.5,ci_97.5,renorm_causal_specificity
0,36,229,0.913869,0.411105,1.435714,0.904908
1,38,60,0.207369,0.062659,0.400115,0.211055
2,27,146,0.100718,0.031798,0.171312,0.088627
3,33,166,0.070097,0.026357,0.107441,0.080556
4,33,206,0.057850,0.013595,0.089700,0.032775


## 22 — Global routing baseline on selection targets

In [24]:
@contextmanager
def capture_routing_for_batch(batch):
    records = {}
    originals = []

    valid_flat = (
        batch["attention_mask"]
        .reshape(-1)
        .bool()
    )

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward

        originals.append(
            (gate, original)
        )

        def make_forward(
            idx,
            original_forward,
        ):
            def patched(
                self,
                hidden_states,
            ):
                logits, weights, selected = (
                    original_forward(
                        hidden_states
                    )
                )

                with torch.no_grad():
                    mask = valid_flat

                    if mask.numel() == selected.shape[0]:
                        mask = mask.to(
                            selected.device
                        )
                    else:
                        mask = torch.ones(
                            selected.shape[0],
                            device=selected.device,
                            dtype=torch.bool,
                        )

                    ids = (
                        selected[mask]
                        .reshape(-1)
                        .long()
                    )

                    ws = (
                        weights[mask]
                        .reshape(-1)
                        .float()
                    )

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )

                    wsum.scatter_add_(
                        0,
                        ids,
                        ws,
                    )

                    records[int(idx)] = {
                        "tokens": int(
                            mask.sum().item()
                        ),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return (
                    logits,
                    weights,
                    selected,
                )

            return patched

        gate.forward = types.MethodType(
            make_forward(
                layer_idx,
                original,
            ),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

In [25]:
selection_target_df = selection_df[
    selection_df["kind"] == "target"
].reset_index(drop=True)

SELECTION_TARGET_BATCH = build_scoring_batch(
    selection_target_df
)

with capture_routing_for_batch(
    SELECTION_TARGET_BATCH
) as routing_records:
    _ = score_batch(
        SELECTION_TARGET_BATCH
    )

routing_rows = []

for layer_idx in SPARSE_LAYERS:
    rec = routing_records[int(layer_idx)]

    tokens = max(
        1,
        rec["tokens"],
    )

    for expert_id in range(
        cfg.num_experts
    ):
        routing_rows.append({
            "layer": int(layer_idx),
            "expert": int(expert_id),
            "selected_rate": float(
                rec["counts"][expert_id].item()
            ) / tokens,
            "routing_mass": float(
                rec["weight_sums"][expert_id].item()
            ) / tokens,
        })

global_routing_df = pd.DataFrame(
    routing_rows
).sort_values(
    ["routing_mass", "selected_rate"],
    ascending=False,
).reset_index(drop=True)

ROUTING_PAIR = (
    int(global_routing_df.iloc[0]["layer"]),
    int(global_routing_df.iloc[0]["expert"]),
)

print("ROUTING_PAIR =", ROUTING_PAIR)

display(
    global_routing_df.head(20)
)

global_routing_df.to_csv(
    RESULTS / "global_routing.csv",
    index=False,
)

ROUTING_PAIR = (29, 194)


,layer,expert,selected_rate,routing_mass
0,29,194,0.649721,0.188190
1,34,228,0.641042,0.163414
2,15,244,0.672350,0.160225
3,25,46,0.686609,0.146864
4,20,34,0.697148,0.142156
5,16,17,0.626782,0.122628
6,6,179,0.703038,0.115344
7,32,240,0.509609,0.098323
8,13,197,0.570986,0.097098
9,3,114,0.630502,0.092475


## 23 — Matched one-expert selector arms

In [26]:
MATCHED_RANDOM_SEED = 2026

rng = np.random.default_rng(
    MATCHED_RANDOM_SEED
)

all_pairs = [
    (int(layer_idx), int(expert_id))
    for layer_idx in SPARSE_LAYERS
    for expert_id in range(cfg.num_experts)
]

random_global_pair = all_pairs[
    int(
        rng.integers(
            0,
            len(all_pairs),
        )
    )
]

same_layer_choices = [
    (CAUSAL_PAIR[0], expert_id)
    for expert_id in range(
        cfg.num_experts
    )
    if expert_id != CAUSAL_PAIR[1]
]

random_same_layer_pair = same_layer_choices[
    int(
        rng.integers(
            0,
            len(same_layer_choices),
        )
    )
]

SELECTOR_ARMS = {
    "causal": CAUSAL_PAIR,
    "routing": ROUTING_PAIR,
    "random_global": random_global_pair,
    "random_same_layer": random_same_layer_pair,
}

selector_df = pd.DataFrame([
    {
        "selector": name,
        "layer": pair[0],
        "expert": pair[1],
    }
    for name, pair in SELECTOR_ARMS.items()
])

display(selector_df)

print(
    "Trainable parameter budget per arm:",
    f"{params_per_expert:,}",
)

,selector,layer,expert
0,causal,36,229
1,routing,29,194
2,random_global,34,56
3,random_same_layer,36,45


Trainable parameter budget per arm: 3,145,728


## 24 — Optional same-layer causal interaction diagnostics

In [27]:
from itertools import combinations

RUN_PAIR_TESTS = True
PAIR_TOP_N = min(
    10,
    len(final_robust),
)

pair_rows = []

if RUN_PAIR_TESTS:
    top = final_robust.head(
        PAIR_TOP_N
    )

    for layer_idx, group in top.groupby("layer"):
        rows = list(
            group.itertuples(index=False)
        )

        for a, b in combinations(
            rows,
            2,
        ):
            m = intervention_score(
                int(layer_idx),
                [
                    int(a.expert),
                    int(b.expert),
                ],
            )

            pair_rows.append({
                "layer": int(layer_idx),
                "expert_a": int(a.expert),
                "expert_b": int(b.expert),
                "pair_causal_specificity": m["causal_specificity"],
                "interaction_score": (
                    m["causal_specificity"]
                    - float(a.causal_specificity)
                    - float(b.causal_specificity)
                ),
            })

pair_df = pd.DataFrame(
    pair_rows
)

if not pair_df.empty:
    pair_df = pair_df.sort_values(
        "pair_causal_specificity",
        ascending=False,
    )

    pair_df.to_csv(
        RESULTS / "pair_interactions.csv",
        index=False,
    )

    display(pair_df)

,layer,expert_a,expert_b,pair_causal_specificity,interaction_score
9,38,60,210,0.234229,0.010555
2,27,146,40,0.136996,0.003727
0,27,146,192,0.136811,-0.010213
1,27,146,118,0.131094,0.005860
6,33,166,206,0.122604,-0.005343
5,27,118,40,0.095670,0.038603
7,33,166,234,0.092939,-0.021884
3,27,192,118,0.067844,-0.002979
8,33,206,234,0.062714,-0.039861
4,27,192,40,0.056687,-0.022171


## 25 — Surgical one-expert bank

In [28]:
import torch.nn as nn
import torch.nn.functional as F

class SurgicalExpertBank(nn.Module):
    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(l), int(e))
            for l, e in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}
        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key = (
                f"L{layer_idx}_E{expert_id}_gu"
            )

            down_key = (
                f"L{layer_idx}_E{expert_id}_down"
            )

            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.params[down_key] = nn.Parameter(
                experts.down_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.key_map[
                (layer_idx, expert_id)
            ] = (
                gu_key,
                down_key,
            )

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        for layer_idx in sorted({
            l for l, _ in self.selected_pairs
        }):
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            original = experts.forward

            self.original_forwards[
                layer_idx
            ] = original

            selected_ids = {
                e
                for l, e in self.selected_pairs
                if l == layer_idx
            }

            bank = self

            def make_forward(
                idx,
                base_experts,
                selected,
            ):
                def surgical_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    final_hidden_states = (
                        torch.zeros_like(
                            hidden_states
                        )
                    )

                    with torch.no_grad():
                        expert_mask = F.one_hot(
                            top_k_index,
                            num_classes=self_experts.num_experts,
                        ).permute(
                            2,
                            1,
                            0,
                        )

                        expert_hit = torch.greater(
                            expert_mask.sum(
                                dim=(-1, -2)
                            ),
                            0,
                        ).nonzero(
                            as_tuple=False
                        ).reshape(-1)

                    for expert_tensor in expert_hit:
                        expert_id = int(
                            expert_tensor.item()
                        )

                        top_k_pos, token_idx = torch.where(
                            expert_mask[
                                expert_id
                            ]
                        )

                        current_state = (
                            hidden_states[
                                token_idx
                            ]
                        )

                        if expert_id in selected:
                            gu_key, down_key = (
                                bank.key_map[
                                    (idx, expert_id)
                                ]
                            )

                            gu = bank.params[
                                gu_key
                            ].to(
                                current_state.dtype
                            )

                            down = bank.params[
                                down_key
                            ].to(
                                current_state.dtype
                            )

                        else:
                            gu = base_experts.gate_up_proj[
                                expert_id
                            ]

                            down = base_experts.down_proj[
                                expert_id
                            ]

                        gate, up = F.linear(
                            current_state,
                            gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        h = (
                            base_experts.act_fn(
                                gate
                            )
                            * up
                        )

                        h = F.linear(
                            h,
                            down,
                        )

                        h = h * top_k_weights[
                            token_idx,
                            top_k_pos,
                            None,
                        ]

                        final_hidden_states.index_add_(
                            0,
                            token_idx,
                            h.to(
                                final_hidden_states.dtype
                            ),
                        )

                    return final_hidden_states

                return surgical_forward

            experts.forward = types.MethodType(
                make_forward(
                    layer_idx,
                    experts,
                    selected_ids,
                ),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original in (
            self.original_forwards.items()
        ):
            get_sparse_mlp(
                layer_idx
            ).experts.forward = original

        self.original_forwards.clear()
        self.installed = False

## 26 — Build target training cases

In [29]:
def make_training_case(
    prompt,
    reference,
    max_length=1024,
):
    prefix_text = chat_prefix_text(
        prompt
    )

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(
            f"{len(full_ids)} tokens > {max_length}"
        )

    start = 0

    for a, b in zip(
        prefix_ids,
        full_ids,
    ):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError(
            "Could not identify answer boundary."
        )

    input_ids = torch.tensor(
        full_ids,
        dtype=torch.long,
        device="cuda:0",
    ).unsqueeze(0)

    attention_mask = torch.ones_like(
        input_ids
    )

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        dtype=torch.long,
        device="cuda:0",
    )

    targets = torch.tensor(
        full_ids[start:],
        dtype=torch.long,
        device="cuda:0",
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

MAX_TRAIN_TOKENS = 1024

TRAIN_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        MAX_TRAIN_TOKENS,
    )
    for r in train_target_df.itertuples(
        index=False
    )
]

print("Training examples:", len(TRAIN_CASES))
print(
    "Token lengths:",
    min(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
    "to",
    max(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
)

Training examples: 50
Token lengths: 66 to 83


## 27 — Matched training configuration

In [30]:
RUN_MATCHED_EXPERIMENT = True

MATCHED_LR = 1e-5
MATCHED_GRAD_ACCUM = 8
MATCHED_EPOCHS = 3
MATCHED_MAX_UPDATES = 50
MATCHED_WEIGHT_DECAY = 0.01

print("RUN_MATCHED_EXPERIMENT =", RUN_MATCHED_EXPERIMENT)

RUN_MATCHED_EXPERIMENT = True


## 28 — Held-out evaluation function

In [31]:
def evaluate_heldout_current_model():
    nll = score_batch(
        HELDOUT_BATCH
    )

    target_mean = float(
        nll[
            heldout_target_mask
        ].mean()
    )

    control_mean = float(
        nll[
            heldout_control_mask
        ].mean()
    )

    target_improvement = float(
        HELDOUT_BASE_NLL[
            heldout_target_mask
        ].mean()
        - target_mean
    )

    control_damage = float(
        control_mean
        - HELDOUT_BASE_NLL[
            heldout_control_mask
        ].mean()
    )

    adaptation_score = (
        target_improvement
        - CONTROL_PENALTY
        * max(
            control_damage,
            0.0,
        )
    )

    return {
        "heldout_target_nll": target_mean,
        "heldout_control_nll": control_mean,
        "target_improvement": target_improvement,
        "control_damage": control_damage,
        "adaptation_score": adaptation_score,
        "per_example_nll": nll,
    }

## 29 — Train one selector arm

In [32]:
def train_one_selector_arm(
    selector_name,
    pair,
):
    layer_idx, expert_id = map(
        int,
        pair,
    )

    bank = SurgicalExpertBank([
        (
            layer_idx,
            expert_id,
        )
    ])

    bank.install()
    bank.train()
    model.train()

    model.config.use_cache = False

    for p in model.parameters():
        p.requires_grad_(False)

    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=MATCHED_LR,
        betas=(0.9, 0.95),
        weight_decay=MATCHED_WEIGHT_DECAY,
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    history = []
    raw_step = 0
    update_step = 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(
            MATCHED_EPOCHS
        ):
            for case_idx, case in enumerate(
                TRAIN_CASES
            ):
                raw_step += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case["input_ids"],
                        attention_mask=case["attention_mask"],
                        use_cache=False,
                        logits_to_keep=case["pred_positions"],
                        return_dict=True,
                    )

                    logits = out.logits.float()

                    loss = F.cross_entropy(
                        logits.reshape(
                            -1,
                            logits.shape[-1],
                        ),
                        case["targets"].reshape(
                            -1
                        ),
                    )

                    scaled = (
                        loss
                        / MATCHED_GRAD_ACCUM
                    )

                scaled.backward()

                final_available_case = (
                    epoch
                    == MATCHED_EPOCHS - 1
                    and case_idx
                    == len(TRAIN_CASES) - 1
                )

                should_step = (
                    raw_step
                    % MATCHED_GRAD_ACCUM
                    == 0
                    or final_available_case
                )

                if should_step:
                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            bank.parameters(),
                            1.0,
                        )
                    )

                    optimizer.step()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "selector": selector_name,
                        "layer": layer_idx,
                        "expert": expert_id,
                        "raw_step": raw_step,
                        "update_step": update_step,
                        "loss": float(
                            loss.detach().item()
                        ),
                        "grad_norm": float(
                            grad_norm
                        ),
                    })

                del out, logits, loss, scaled

                if (
                    update_step
                    >= MATCHED_MAX_UPDATES
                ):
                    break

            if (
                update_step
                >= MATCHED_MAX_UPDATES
            ):
                break

        model.eval()
        bank.eval()

        metrics = (
            evaluate_heldout_current_model()
        )

        bank_state_cpu = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        result = {
            "selector": selector_name,
            "layer": layer_idx,
            "expert": expert_id,
            "trainable_params": bank.trainable_parameter_count,
            "updates": update_step,
            "final_train_loss": (
                history[-1]["loss"]
                if history
                else np.nan
            ),
            "peak_gpu_gib": (
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            "heldout_target_nll": metrics["heldout_target_nll"],
            "heldout_control_nll": metrics["heldout_control_nll"],
            "target_improvement": metrics["target_improvement"],
            "control_damage": metrics["control_damage"],
            "adaptation_score": metrics["adaptation_score"],
            "history": history,
            "bank_state_cpu": bank_state_cpu,
            "per_example_nll": metrics["per_example_nll"],
        }

        return result

    finally:
        bank.restore()

        if hasattr(
            model,
            "disable_input_require_grads",
        ):
            model.disable_input_require_grads()

        try:
            model.gradient_checkpointing_disable()
        except Exception:
            pass

        model.eval()

        del optimizer
        del bank

        gc.collect()
        torch.cuda.empty_cache()

## 30 — Run matched one-expert experiment

In [33]:
MATCHED_RESULTS = {}
MATCHED_RESULT_ROWS = []
MATCHED_HISTORY_ROWS = []

if RUN_MATCHED_EXPERIMENT:
    for selector_name, pair in (
        SELECTOR_ARMS.items()
    ):
        print("\n" + "=" * 72)
        print(
            "Training selector:",
            selector_name,
            "pair:",
            pair,
        )
        print("=" * 72)

        result = train_one_selector_arm(
            selector_name,
            pair,
        )

        MATCHED_RESULTS[
            selector_name
        ] = result

        MATCHED_RESULT_ROWS.append({
            k: v
            for k, v in result.items()
            if k not in {
                "history",
                "bank_state_cpu",
                "per_example_nll",
            }
        })

        MATCHED_HISTORY_ROWS.extend(
            result["history"]
        )

        print(
            f"{selector_name}: "
            f"target improvement="
            f"{result['target_improvement']:+.4f}, "
            f"control damage="
            f"{result['control_damage']:+.4f}, "
            f"adaptation score="
            f"{result['adaptation_score']:+.4f}"
        )

    matched_results_df = pd.DataFrame(
        MATCHED_RESULT_ROWS
    ).sort_values(
        "adaptation_score",
        ascending=False,
    ).reset_index(drop=True)

    matched_history_df = pd.DataFrame(
        MATCHED_HISTORY_ROWS
    )

    display(matched_results_df)

else:
    print("Matched experiment disabled.")


Training selector: causal pair: (36, 229)
causal: target improvement=+0.0280, control damage=-0.0337, adaptation score=+0.0280

Training selector: routing pair: (29, 194)
routing: target improvement=+0.7719, control damage=-0.6209, adaptation score=+0.7719

Training selector: random_global pair: (34, 56)
random_global: target improvement=+0.0031, control damage=-0.0022, adaptation score=+0.0031

Training selector: random_same_layer pair: (36, 45)
random_same_layer: target improvement=+0.0171, control damage=-0.0178, adaptation score=+0.0171


,selector,layer,expert,trainable_params,updates,final_train_loss,peak_gpu_gib,heldout_target_nll,heldout_control_nll,target_improvement,control_damage,adaptation_score
0,routing,29,194,3145728,19,4.423398,63.622203,3.990968,3.499274,0.771936,-0.620873,0.771936
1,causal,36,229,3145728,19,4.936981,63.622203,4.734897,4.086417,0.028006,-0.033730,0.028006
2,random_same_layer,36,45,3145728,19,4.904040,63.622203,4.745832,4.102362,0.017071,-0.017785,0.017071
3,random_global,34,56,3145728,19,5.127804,63.622203,4.759829,4.117951,0.003075,-0.002196,0.003075


## 31 — Save results and compact one-expert banks

In [34]:
if RUN_MATCHED_EXPERIMENT:
    MATCHED_DIR = (
        RESULTS
        / "matched_one_expert_experiment"
    )

    MATCHED_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    matched_results_df.to_csv(
        MATCHED_DIR
        / "matched_selector_results.csv",
        index=False,
    )

    matched_history_df.to_csv(
        MATCHED_DIR
        / "matched_training_history.csv",
        index=False,
    )

    selector_df.to_csv(
        MATCHED_DIR
        / "selector_arms.csv",
        index=False,
    )

    for selector_name, result in (
        MATCHED_RESULTS.items()
    ):
        torch.save(
            {
                "model_id": MODEL_ID,
                "selector": selector_name,
                "layer": result["layer"],
                "expert": result["expert"],
                "trainable_params": result["trainable_params"],
                "learning_rate": MATCHED_LR,
                "grad_accum": MATCHED_GRAD_ACCUM,
                "updates": result["updates"],
                "state_dict": result["bank_state_cpu"],
            },
            MATCHED_DIR
            / f"{selector_name}_expert_bank.pt",
        )

        np.save(
            MATCHED_DIR
            / f"{selector_name}_heldout_nll.npy",
            result["per_example_nll"],
        )

    print("Saved matched experiment:", MATCHED_DIR)

Saved matched experiment: /home/ec2-user/workspace/laguna_xs2_v5_results/matched_one_expert_experiment


## 32 — Final experiment summary

In [35]:
summary = {
    "causal_pair": CAUSAL_PAIR,
    "routing_pair": ROUTING_PAIR,
    "random_global_pair": SELECTOR_ARMS["random_global"],
    "random_same_layer_pair": SELECTOR_ARMS["random_same_layer"],
    "params_per_arm": int(params_per_expert),
    "selection_examples": int(len(selection_df)),
    "train_examples": int(len(train_target_df)),
    "heldout_examples": int(len(heldout_df)),
}

print(summary)

print("\nTop causal experts:")
display(
    strict_candidates[
        [
            "layer",
            "expert",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "renorm_causal_specificity",
        ]
    ].head(10)
)

if RUN_MATCHED_EXPERIMENT:
    print("\nMatched adaptation result:")
    display(
        matched_results_df[
            [
                "selector",
                "layer",
                "expert",
                "trainable_params",
                "updates",
                "target_improvement",
                "control_damage",
                "adaptation_score",
                "peak_gpu_gib",
            ]
        ]
    )

{'causal_pair': (36, 229), 'routing_pair': (29, 194), 'random_global_pair': (34, 56), 'random_same_layer_pair': (36, 45), 'params_per_arm': 3145728, 'selection_examples': 100, 'train_examples': 50, 'heldout_examples': 100}

Top causal experts:


,layer,expert,causal_specificity,ci_2.5,ci_97.5,renorm_causal_specificity
0,36,229,0.913869,0.411105,1.435714,0.904908
1,38,60,0.207369,0.062659,0.400115,0.211055
2,27,146,0.100718,0.031798,0.171312,0.088627
3,33,166,0.070097,0.026357,0.107441,0.080556
4,33,206,0.057850,0.013595,0.089700,0.032775



Matched adaptation result:


,selector,layer,expert,trainable_params,updates,target_improvement,control_damage,adaptation_score,peak_gpu_gib
0,routing,29,194,3145728,19,0.771936,-0.620873,0.771936,63.622203
1,causal,36,229,3145728,19,0.028006,-0.033730,0.028006,63.622203
2,random_same_layer,36,45,3145728,19,0.017071,-0.017785,0.017071,63.622203
3,random_global,34,56,3145728,19,0.003075,-0.002196,0.003075,63.622203


## 33 — Manifest + archive

In [36]:
import json
from datetime import datetime, timezone

manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_id": MODEL_ID,
    "model_path": str(MODEL_PATH),
    "experiment_csv": str(EXPERIMENT_CSV),
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_vram_gib": (
            torch.cuda.get_device_properties(0)
            .total_memory
            / 2**30
        ),
        "logical_cpus": os.cpu_count(),
        "ram_gib": (
            psutil.virtual_memory().total
            / 2**30
        ),
    },
    "architecture": {
        "layers": int(cfg.num_hidden_layers),
        "sparse_layers": list(
            map(int, SPARSE_LAYERS)
        ),
        "experts": int(cfg.num_experts),
        "top_k": int(cfg.num_experts_per_tok),
        "expert_width": int(
            cfg.moe_intermediate_size
        ),
        "params_per_expert": int(
            params_per_expert
        ),
    },
    "search": {
        "top_layers": TOP_LAYERS,
        "search_seeds": SEARCH_SEEDS,
        "initial_group_size": INITIAL_GROUP_SIZE,
        "beam_width": BEAM_WIDTH,
        "control_penalty": CONTROL_PENALTY,
    },
    "selectors": {
        k: list(map(int, v))
        for k, v in SELECTOR_ARMS.items()
    },
    "matched_training": {
        "enabled": RUN_MATCHED_EXPERIMENT,
        "learning_rate": MATCHED_LR,
        "grad_accum": MATCHED_GRAD_ACCUM,
        "epochs": MATCHED_EPOCHS,
        "max_updates": MATCHED_MAX_UPDATES,
        "weight_decay": MATCHED_WEIGHT_DECAY,
    },
}

(RESULTS / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

archive = shutil.make_archive(
    str(RESULTS),
    "zip",
    root_dir=RESULTS,
)

print("Results directory:", RESULTS)
print("Archive:", archive)

Results directory: /home/ec2-user/workspace/laguna_xs2_v5_results
Archive: /home/ec2-user/workspace/laguna_xs2_v5_results.zip
